We will add some new properties to the `entity` object to allow it to move about a plane to find food. With this addition, we can simulate a population growth contrained by limited available food resources. 

In [2]:
# Import required modules
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from random import random, randint
from collections import Counter, namedtuple
from math import hypot

Collision detection:

2 objects
https://www.youtube.com/watch?v=XYzA_kPWyJ8
multiple objects
https://www.youtube.com/watch?v=789weryntzM


In [3]:
class Vector:
    """ A vector stores information and operations regarding the 
        direction and speed of motion."""

    def __init__(self, vx, vy):
        self.vx = vx
        self.vy = vy

    def __repr__(self):
        return 'Vector(%r, %r)' % (self.vx, self.vy)

    def __abs__(self):
        return hyxpot(self.vx, self.vy)

    def __bool__(self):
        return bool(abs(self))

    def __add__(self, other):
        vx = self.vx + other.vx
        vy = self.vy + other.vy
        return Vector(vx, vy)

    def __mul__(self, scalar):
        return Vector(self.vx * scalar, self.vy * scalar)

        

In [3]:
Extent = namedtuple("area", ["width", "height"])

def trial(chance):
    """ Returns the boolean outcome of a trial given it's chance of success."""

    return random() <= chance


class Entity():
    """ The entity object represents a live entity."""
    
    def __init__(self, x, y):
        self.period = 0
        self.x = x
        self.y = y
    
    def __repr__(self):
        return str(self.period)

    def __call__(self):
        self.period += 1
        return self
    

class Environment():
    """ The environment tracks the population and parameters of the simulation.

            Parameters:
                repro_chance (float):               Chance of reproduction
                death_chance (float):               Chance of death
                starting_pop (int):                 Starting population
                extent (iterable [width, height]):  Extent of habitat
    """

    def __init__(self, repro_chance, death_chance, starting_pop, extent):
        self.repro_chance = repro_chance
        self.death_chance = death_chance
        self.starting_pop = starting_pop
        self.extent = Extent(*extent) #might turn this into a habitat class

        self.population = [
            Entity(randint(0, self.extent.width), randint(0, self.extent.height))
            for p in range(0, starting_pop)
        ]
        self.history = [starting_pop]
        self.mortuary = Counter()

    def __call__(self):

        # Simulate period
        survive, expire, births = [], [], []
        # First determine which entities expired by end of previous period.
        for e in self.population:
            if trial(self.death_chance):
               expire.append(e().period) 
            else:
                survive.append(e())
                # Determine if the entity reproduced by end of current period.
                if trial(self.repro_chance):
                    births.append(Entity(randint(0, self.extent.width), randint(0, self.extent.height)))

        # Finally, update attributes
        self.population = survive + births
        self.mortuary.update(expire)
        self.history.append(len(self.population))